# Kaggle Spoken Grammar Scoring Engine

**Objective**: Predict continuous English grammar proficiency scores from `0.0` to `5.0` using spoken audio recordings.

- **Target Metric**: Root Mean Squared Error (RMSE) and Pearson Correlation ($r$).
- **Aspirational Benchmark**: Out-Of-Fold (OOF) RMSE $\le$ 0.40.
- **Dataset Structure**:
  - 769 labelled training recordings (`train.csv` + WAV files).
  - 216 test recordings (`test.csv` + WAV files; placeholder target `-1.0`).
  - Sample submission contract (`sample_submission.csv`).

---
### Pipeline Architectural Principles
1. **Leak-Free Cross-Validation**: Strict 5-Fold splitting (GroupKFold if speaker IDs detected). Supervised scalers and feature selectors fit strictly inside fold loops.
2. **Non-Destructive Preprocessing**: Signal properties verified before transformation; no arbitrary filtering without evidence.
3. **Persistent Checkpointing & Atomic I/O**: Checkpoints and feature caches written via atomic renames to prevent corruption.
4. **No Fabricated Data**: Never invent statistics or run results without live execution on the Kaggle environment.

## 1. Environment Setup & Configuration

This cell dynamically checks the environment (Kaggle vs. Local development), configures random seeds for absolute reproducibility, and initializes directory paths.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure repository src modules are discoverable
REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "src").exists():
    sys.path.insert(0, str(REPO_ROOT))
elif (Path.cwd().parent / "src").exists():
    sys.path.insert(0, str(Path.cwd().parent))
    REPO_ROOT = Path.cwd().parent

import random
import numpy as np
import pandas as pd

from src.config import ProjectConfig, is_kaggle_environment, setup_logging

# Set up logging
setup_logging()

# Load centralized project configuration with automatic Kaggle directory mapping
config = ProjectConfig.load(auto_kaggle=True)
config.paths.ensure_directories()

# Seed reproducibility
SEED = config.random_seed
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Environment: {'Kaggle' if is_kaggle_environment() else 'Local'}")
print(f"Data Directory: {config.paths.data_dir}")
print(f"Outputs Directory: {config.paths.output_dir}")
print(f"Features Cache Directory: {config.paths.cache_dir}")

## 2. Phase 1: Comprehensive Dataset & Audio Signal Audit

The audit inspects the physical CSV files, identifies column formats, computes target distribution statistics, and verifies the WAV files (sample rates, channels, clipping, silence, and file corruption) without modifying any data.

In [ ]:
from IPython.display import Markdown, display
from src.audio_audit import run_full_dataset_audit
from src.artifacts import atomic_save_json, atomic_write_text

train_csv = config.paths.train_csv_path
test_csv = config.paths.test_csv_path
sample_sub = config.paths.sample_submission_path

assert train_csv.exists(), f"Train CSV not found at {train_csv}. Please verify Kaggle dataset attachment."
assert test_csv.exists(), f"Test CSV not found at {test_csv}. Please verify Kaggle dataset attachment."

sample_sub_path = sample_sub if sample_sub.exists() else None

print("Running comprehensive dataset audit...")
report = run_full_dataset_audit(
    train_csv_path=train_csv,
    test_csv_path=test_csv,
    sample_sub_path=sample_sub_path,
    audio_dirs_train=[config.paths.train_audio_path, config.paths.data_dir],
    audio_dirs_test=[config.paths.test_audio_path, config.paths.data_dir],
    silence_threshold_db=config.audio.silence_threshold_db,
    clipping_threshold=config.audio.clipping_threshold,
)

# Persist audit artifacts atomically
audit_json_path = config.paths.output_dir / "audit_report.json"
audit_md_path = config.paths.output_dir / "audit_report.md"
atomic_save_json(audit_json_path, report.to_dict())
atomic_write_text(audit_md_path, report.to_markdown())

# Display markdown report in notebook
display(Markdown(report.to_markdown()))

### Target Distribution & Tabular Diagnostics Visualizer

Inspect the distribution of grammar proficiency scores in `train.csv`. Confirm that target values reside within $[0.0, 5.0]$ and note whether scores are continuous or graded on discrete rubrics (e.g. 0.5 increments).

In [ ]:
import matplotlib.pyplot as plt

train_df = pd.read_csv(train_csv)
target_col = report.target.target_column_name if report.target else "score"

if target_col in train_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram and KDE
    scores = train_df[target_col].dropna()
    axes[0].hist(scores, bins=20, color="#2b5c8f", edgecolor="white", alpha=0.8, density=True)
    scores.plot.kde(ax=axes[0], color="#e76f51", lw=2)
    axes[0].set_title(f"Grammar Score Distribution (N={len(scores)})")
    axes[0].set_xlabel("Score (0.0 to 5.0)")
    axes[0].set_ylabel("Density")
    axes[0].set_xlim(0, 5.2)
    axes[0].grid(True, linestyle="--", alpha=0.5)
    
    # Box plot
    axes[1].boxplot(scores, vert=False, patch_artist=True, boxprops=dict(facecolor="#2a9d8f", color="#264653"))
    axes[1].set_title("Grammar Score Boxplot & Outlier Detection")
    axes[1].set_xlabel("Score")
    axes[1].grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Mean Score: {scores.mean():.4f} | Std: {scores.std():.4f} | Median: {scores.median():.4f}")
    print(f"Minimum: {scores.min():.4f} | Maximum: {scores.max():.4f}")
else:
    print(f"Target column '{target_col}' not found in train.csv")

## 3. Phase 2: Audio Preprocessing Pipeline (Specifications)

Before passing waveforms into pretrained encoders (e.g., Wav2Vec2/HuBERT):
1. **Sampling Rate**: Resample to 16,000 Hz if recorded at different rates.
2. **Channel Format**: Convert stereo to mono via channel averaging.
3. **Duration Handling**: Truncate/pad to encoder input specifications without arbitrary loss of speech content.
4. **No Destructive Filtering**: Preserves speech pauses and natural disfluencies (critical indicators of spoken grammar competence).

## 4. Phase 3: Validation Strategy & Leakage Prevention

Cross-validation safeguards:
- 5-fold cross-validation with fixed seed.
- Group-based splitting if speaker IDs are identified in the audit report.
- Every sample evaluated out-of-fold exactly once.

In [ ]:
from src.validation import create_folds, verify_fold_integrity, save_fold_assignments

# Check if speaker metadata was detected during the audit
group_col = report.tabular.speaker_metadata.get("column") if report.tabular.speaker_metadata else None
fold_df = create_folds(
    train_df,
    target_col=target_col,
    group_col=group_col,
    n_splits=config.validation.n_splits,
    seed=config.random_seed,
)

# Verify integrity and absence of data leakage
fold_report = verify_fold_integrity(fold_df, group_col=group_col)
print(f"Fold Integrity Valid: {fold_report.is_valid}")
print(f"Samples per fold: {fold_report.samples_per_fold}")
if fold_report.leakage_errors:
    print(f"Leakage errors: {fold_report.leakage_errors}")

# Persist fold assignments to disk
folds_csv_path = config.paths.output_dir / "folds.csv"
id_col = report.tabular.train_columns[0]
save_fold_assignments(fold_df, folds_csv_path, id_col=id_col)
print(f"Saved verified fold assignments to {folds_csv_path}")

## 5. Phase 4: Feature Extraction Architecture (Audio Embeddings)

Embeddings are extracted using a single pretrained speech encoder (`facebook/wav2vec2-base-960h`), pooled into fixed-size representations, and cached atomically.

## 6. Phase 5: Modeling & Incremental Benchmarks

We evaluate models incrementally:
1. **Mean Prediction Baseline** (establishes empirical floor)
2. **Audio Embeddings + Ridge Regression** (regularized linear baseline)
3. **Audio Embeddings + Support Vector Regression (SVR)** (non-linear kernel)

## 7. Phase 6: Model Comparison, Diagnostics & Submission Safeguards

Final model comparison, residual analysis, test prediction clipping to $[0.0, 5.0]$, and submission alignment checks.

### 4.1 Mean Baseline Benchmark (Empirical Floor)

Evaluates the minimal performance floor predicting the training target mean across all 5 folds.

In [ ]:
from src.models import MeanBaselineModel
from src.metrics import root_mean_squared_error, pearson_correlation, mean_absolute_error

print('=== Evaluating Mean Baseline Model (Empirical Floor) ===')
y_train_all = fold_df[target_col].to_numpy(dtype=float)
mean_oof_preds = np.zeros(len(fold_df))

for f in range(5):
    val_m = (fold_df['fold'] == f).to_numpy()
    train_y = y_train_all[~val_m]
    baseline_m = MeanBaselineModel(strategy='mean')
    baseline_m.fit(None, train_y)
    mean_oof_preds[val_m] = baseline_m.predict(y_train_all[val_m])

mean_rmse = root_mean_squared_error(y_train_all, mean_oof_preds)
mean_r = pearson_correlation(y_train_all, mean_oof_preds)
mean_mae = mean_absolute_error(y_train_all, mean_oof_preds)
print(f'Mean Baseline OOF RMSE: {mean_rmse:.4f}')
print(f'Mean Baseline Pearson r: {mean_r:.4f}')
print(f'Mean Baseline MAE:       {mean_mae:.4f}')


In [ ]:
from src.data import load_dataset
from src.features import extract_dataset_features

print('=== Extracting Audio Features (with atomic caching) ===')
train_data, _ = load_dataset(train_csv, audio_dirs=[config.paths.train_audio_path, config.paths.data_dir])
test_data, test_map = load_dataset(test_csv, audio_dirs=[config.paths.test_audio_path, config.paths.data_dir], is_test=True)

if 'resolved_audio_path' not in fold_df.columns:
    fold_df['resolved_audio_path'] = train_data['resolved_audio_path']

cache_dir = config.paths.cache_dir
cache_dir.mkdir(parents=True, exist_ok=True)
train_cache = cache_dir / 'train_features.npy'
test_cache = cache_dir / 'test_features.npy'

try:
    import torch
    method = 'wav2vec2' if torch.cuda.is_available() else 'acoustic'
except ImportError:
    method = 'acoustic'

print(f'Feature extraction method selected: {method}')
X_train = extract_dataset_features(fold_df['resolved_audio_path'].tolist(), method=method, cache_path=train_cache)
X_test = extract_dataset_features(test_data['resolved_audio_path'].tolist(), method=method, cache_path=test_cache)
print(f'Feature matrix extracted -> X_train: {X_train.shape} | X_test: {X_test.shape}')


In [ ]:
from src.models import RidgeGrammarModel, SVREngineModel, run_cv_experiment

print('=== Training Ridge Regression Model ===')
ridge_res = run_cv_experiment(
    fold_df=fold_df,
    features=X_train,
    target_col=target_col,
    model_factory=lambda: RidgeGrammarModel(),
    model_name='Ridge Regression',
)
print(f'Ridge OOF RMSE: {ridge_res["oof_rmse"]:.4f} | Pearson r: {ridge_res["oof_pearson_r"]:.4f}')

print('\n=== Training SVR (RBF Kernel) Model ===')
svr_res = run_cv_experiment(
    fold_df=fold_df,
    features=X_train,
    target_col=target_col,
    model_factory=lambda: SVREngineModel(C=1.0, epsilon=0.1),
    model_name='SVR (RBF Kernel)',
)
print(f'SVR OOF RMSE:   {svr_res["oof_rmse"]:.4f} | Pearson r: {svr_res["oof_pearson_r"]:.4f}')


In [ ]:
# Model Benchmarking Summary Table
results_summary = pd.DataFrame([
    {'Model': 'Mean Baseline', 'OOF RMSE': round(mean_rmse, 4), 'Pearson r': round(mean_r, 4), 'MAE': round(mean_mae, 4)},
    {'Model': 'Ridge Regression', 'OOF RMSE': round(ridge_res['oof_rmse'], 4), 'Pearson r': round(ridge_res['oof_pearson_r'], 4), 'MAE': round(ridge_res['oof_mae'], 4)},
    {'Model': 'SVR (RBF Kernel)', 'OOF RMSE': round(svr_res['oof_rmse'], 4), 'Pearson r': round(svr_res['oof_pearson_r'], 4), 'MAE': round(svr_res['oof_mae'], 4)},
])
display(results_summary)

# Select best model based on OOF RMSE
best_res = ridge_res if ridge_res['oof_rmse'] <= svr_res['oof_rmse'] else svr_res
print(f'\nBest Selected Model: {best_res["model_name"]} (OOF RMSE: {best_res["oof_rmse"]:.4f})')

# Generate ensemble 5-fold test predictions
test_preds = np.mean([model.predict(X_test) for model in best_res['models']], axis=0)
test_preds = np.clip(test_preds, 0.0, 5.0)

# Generate and save submission.csv
submission_df = pd.DataFrame({
    test_map.filename_col: test_data[test_map.filename_col],
    target_col: test_preds,
})
submission_path = config.paths.output_dir / 'submission.csv'
submission_df.to_csv(submission_path, index=False)
print(f'Saved final submission to {submission_path} (Shape: {submission_df.shape})')
display(submission_df.head(10))
